# 🧬 LangGraph Computational Pathology V5
## Reviewer Reproducibility Notebook

**No installation needed — just run all cells top-to-bottom (Runtime → Run all).**

This notebook reproduces the full TCGA-PAAD spatiotemporal pipeline on the
bundled open-access H&E whole-slide image, then plots the key figures.

**Estimated runtime:** ~4 minutes on Colab free tier (CPU).

---
| Field | Value |
|---|---|
| Manuscript | *LLM-Orchestrated Spatiotemporal Reconstruction of the TME from H&E WSIs* |
| Repository | https://github.com/bayjuan5/LangGraphPrj_V5 |
| Contact | bbh@imdlab.org |
| ORCID | [0000-0003-0555-6924](https://orcid.org/0000-0003-0555-6924) |

## Step 1 — Install dependencies & clone repository

In [ ]:
# Install system OpenSlide library (required for SVS reading)
!apt-get install -y -q libopenslide0 libopenslide-dev 2>/dev/null

# Install Python packages
!pip install -q openslide-python openslide-bin scikit-image scikit-learn \
    matplotlib pillow numpy tqdm

print('✅ Dependencies installed')

In [ ]:
import os, sys

# Clone the repository (skip if already present)
if not os.path.exists('LangGraphPrj_V5'):
    !git clone -q https://github.com/bayjuan5/LangGraphPrj_V5.git
    print('✅ Repository cloned')
else:
    !git -C LangGraphPrj_V5 pull -q
    print('✅ Repository up to date')

# Add to Python path so we can import nodes directly
sys.path.insert(0, 'LangGraphPrj_V5')

SVS_PATH = ('LangGraphPrj_V5/TCGA_test/'
            'TCGA-HZ-7926-01Z-00-DX1.'
            'b3bf02d3-bad0-4451-9c39-b0593f19154c.svs')

print(f'SVS file present: {os.path.exists(SVS_PATH)}')
print(f'SVS size: {os.path.getsize(SVS_PATH)/1e6:.1f} MB')

## Step 2 — Inspect the TCGA-PAAD whole-slide image

In [ ]:
import openslide
import matplotlib.pyplot as plt
import numpy as np

slide = openslide.OpenSlide(SVS_PATH)
w, h  = slide.dimensions

print(f'Case        : TCGA-HZ-7926 (Pancreatic Adenocarcinoma)')
print(f'Stain       : H&E')
print(f'Dimensions  : {w:,} x {h:,} px')
print(f'Levels      : {slide.level_count}  (downsamples: {[round(d,1) for d in slide.level_downsamples]})')

# Show thumbnail
thumb = slide.get_thumbnail((800, 600))
slide.close()

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(thumb)
ax.set_title('TCGA-HZ-7926 — H&E Whole-Slide Image (thumbnail)', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()
print('\u2705 WSI loaded and displayed')

## Step 3 — Node 1: Adaptive WSI Tiling

In [ ]:
from nodes.node1_tiling import process as node1

state = {
    'svs_path':         SVS_PATH,
    'patch_size':       512,
    'tissue_threshold': 0.40,
    'max_tiles':        300,
}

state = node1(state)
print(f'\n--- Node 1 Output ---')
print(f'Tiles extracted : {state["n_tiles"]}')
print(f'WSI dimensions  : {state["wsi_dims"][0]:,} x {state["wsi_dims"][1]:,} px')

In [ ]:
# Visualise tile locations on the thumbnail
import openslide
slide  = openslide.OpenSlide(SVS_PATH)
thumb  = np.array(slide.get_thumbnail((800, 600)))
slide.close()

scale_x = 800 / state['wsi_dims'][0]
scale_y = 600 / state['wsi_dims'][1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(thumb)
for t in state['tiles']:
    px = t['x'] * scale_x
    py = t['y'] * scale_y
    rect = plt.Rectangle((px, py),
                          state['patch_size'] * scale_x,
                          state['patch_size'] * scale_y,
                          linewidth=0.4, edgecolor='lime', facecolor='none', alpha=0.6)
    axes[0].add_patch(rect)
axes[0].set_title(f'Tissue Tiles (n={state["n_tiles"]})', fontweight='bold')
axes[0].axis('off')

fracs = [t['tissue_fraction'] for t in state['tiles']]
axes[1].hist(fracs, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(0.40, color='red', linestyle='--', label='threshold=0.40')
axes[1].set_xlabel('Tissue Fraction per Tile')
axes[1].set_ylabel('Count')
axes[1].set_title('Tissue Fraction Distribution')
axes[1].legend()

plt.suptitle('Node 1 — Adaptive WSI Tiling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4 — Node 2.1: ROSIE Biomarker Inference (50-channel protein matrix)

In [ ]:
from nodes.node2_1_rosie import process as node21
import warnings; warnings.filterwarnings('ignore')

state = node21(state)
print(f'\n--- Node 2.1 Output ---')
print(f'Protein matrix  : {len(state["protein_matrix"])} tiles x {state["n_channels"]} channels')
print(f'Immune score    : {state["immune_score"]:.3f}  (T-cell panel mean)')
print(f'Stromal score   : {state["stromal_score"]:.3f}  (fibrosis panel mean)')

In [ ]:
import numpy as np

pm     = np.array(state['protein_matrix'])
ch     = state['protein_channels']
means  = pm.mean(axis=0)
stds   = pm.std(axis=0)

# Group indices
groups = {
    'Immune/T-cell':       list(range(0, 5)),
    'Macrophage/Myeloid':  list(range(5, 10)),
    'B-cell/NK':           list(range(10, 15)),
    'Tumour/Epithelial':   list(range(15, 20)),
    'Stromal/Fibroblast':  list(range(20, 25)),
    'Proliferation':       list(range(25, 30)),
    'Apoptosis/Stress':    list(range(30, 35)),
    'Checkpoints':         list(range(35, 40)),
    'Cytokines':           list(range(40, 45)),
    'Angiogenesis':        list(range(45, 50)),
}

group_means = {g: means[idx].mean() for g, idx in groups.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-group bar chart
g_names = list(group_means.keys())
g_vals  = list(group_means.values())
colors  = ['#e74c3c' if 'Immune' in g or 'Check' in g or 'Cyto' in g
           else '#3498db' if 'Stromal' in g or 'Angio' in g
           else '#2ecc71' if 'Tumour' in g or 'Prolif' in g
           else '#95a5a6' for g in g_names]
bars = axes[0].barh(g_names, g_vals, color=colors, edgecolor='white')
axes[0].axvline(state['immune_score'], color='red',  linestyle='--', linewidth=1.5, label=f'Immune score={state["immune_score"]:.2f}')
axes[0].axvline(state['stromal_score'], color='blue', linestyle='--', linewidth=1.5, label=f'Stromal score={state["stromal_score"]:.2f}')
axes[0].set_xlabel('Mean Expression (0–1)')
axes[0].set_title('Protein Expression by Pathway Group')
axes[0].legend(fontsize=8)

# Right: heatmap (tiles x selected proteins)
selected = [0,1,2,3,15,20,21,25,35,36]
sel_names = [ch[i] for i in selected]
im = axes[1].imshow(pm[:50, selected].T, aspect='auto', cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].set_yticks(range(len(sel_names))); axes[1].set_yticklabels(sel_names)
axes[1].set_xlabel('Tile index (first 50)')
axes[1].set_title('Protein Heatmap (selected channels)')
plt.colorbar(im, ax=axes[1], label='Expression')

plt.suptitle('Node 2.1 — ROSIE Biomarker Inference', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Node 2.2: HED Segmentation (Nuclear Morphology)

In [ ]:
from nodes.node2_2_hed import process as node22

state = node22(state)
print(f'\n--- Node 2.2 Output ---')
print(f'Nuclei detected      : {state["n_cells"]}')
print(f'Mean nuclear area    : {state["mean_nuclear_area"]:.1f} px\u00b2')
print(f'Mean eccentricity    : {state["mean_eccentricity"]:.3f}')
print(f'Cell density         : {state["cell_density_per_mm2"]:.1f} cells/mm\u00b2')

In [ ]:
import openslide
from nodes.node2_2_hed import _hed_deconvolve, _segment_nuclei
from skimage import measure, color

# Show HED deconvolution on one representative patch
slide = openslide.OpenSlide(SVS_PATH)
tile  = state['tiles'][0]
rgb   = np.array(slide.read_region((tile['x'], tile['y']), 0, (512, 512)).convert('RGB'))
slide.close()

hed    = _hed_deconvolve(rgb)
labels = _segment_nuclei(hed[..., 0])
overlay = color.label2rgb(labels, image=rgb, bg_label=0, alpha=0.35)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(rgb);             axes[0].set_title('Original H&E');         axes[0].axis('off')
axes[1].imshow(hed[...,0], cmap='Blues');   axes[1].set_title('Haematoxylin (nuclear)'); axes[1].axis('off')
axes[2].imshow(hed[...,1], cmap='Reds');    axes[2].set_title('Eosin (cytoplasm)');      axes[2].axis('off')
axes[3].imshow(overlay);         axes[3].set_title(f'Segmented nuclei (n={labels.max()})'); axes[3].axis('off')

plt.suptitle('Node 2.2 — HED Colour Deconvolution + Nucleus Segmentation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Cell morphology distributions
cf   = state['cell_features']
areas = [c['area'] for c in cf]
eccs  = [c['eccentricity'] for c in cf]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(areas, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Nuclear Area (px\u00b2)'); axes[0].set_ylabel('Count')
axes[0].set_title(f'Nuclear Area Distribution  (mean={state["mean_nuclear_area"]:.0f} px\u00b2)')
axes[1].hist(eccs, bins=30, color='coral', edgecolor='white')
axes[1].set_xlabel('Eccentricity'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Nuclear Eccentricity  (mean={state["mean_eccentricity"]:.3f})')
plt.tight_layout()
plt.show()

## Step 6 — Node 3: Timed Petri Net — Temporal Immune Trajectory

In [ ]:
from nodes.node3_petri_net import process as node3, PLACES, TIMEPOINTS

state = node3(state)
summary = state['tpn_summary']
print(f'\n--- Node 3 Output ---')
print(f'Timepoints modelled   : {summary["n_timepoints"]} (weeks {TIMEPOINTS[0]}–{TIMEPOINTS[-1]})')
print(f'Early immune activation : {summary["early_immune_activation"]:.3f}')
print(f'Mid myeloid transition  : {summary["mid_myeloid_transition"]:.3f}')
print(f'Late stromal dominance  : {summary["late_stromal_dominance"]:.3f}')

In [ ]:
traj = state['temporal_trajectory']
weeks = [int(k.split('_')[1]) for k in sorted(traj.keys(), key=lambda x: int(x.split('_')[1]))]

# Matrix: timepoints x places
mat = np.array([[traj[f'week_{w}'][p] for p in PLACES] for w in weeks])

cmap_colors = ['#2ecc71','#27ae60','#c0392b','#8e44ad',
               '#e67e22','#e74c3c','#3498db','#95a5a6']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Stacked area chart
axes[0].stackplot(weeks, mat.T, labels=PLACES, colors=cmap_colors, alpha=0.85)
axes[0].set_xlabel('Weeks post tumour initiation')
axes[0].set_ylabel('Cell-state fraction')
axes[0].set_title('Temporal Immune Trajectory (Petri Net)')
axes[0].legend(loc='upper left', fontsize=7, ncol=2)
axes[0].set_xticks(weeks)

# Heatmap
im = axes[1].imshow(mat.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=mat.max())
axes[1].set_xticks(range(len(weeks))); axes[1].set_xticklabels([f'Wk{w}' for w in weeks])
axes[1].set_yticks(range(len(PLACES))); axes[1].set_yticklabels(PLACES)
axes[1].set_title('Cell-state Fraction Heatmap')
plt.colorbar(im, ax=axes[1], label='Fraction')

# Annotate phase boundaries
for ax in axes:
    ax.axvspan(4, 6.5, alpha=0.07, color='green', label='Early')
    ax.axvspan(6.5, 9.5, alpha=0.07, color='orange', label='Transitional')
    ax.axvspan(9.5, 12, alpha=0.07, color='red', label='Late')

plt.suptitle('Node 3 — Timed Petri Net: 8-Timepoint Immune Trajectory', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7 — Node 4: Spatial Niche Analysis (DBSCAN + KMeans)

In [ ]:
from nodes.node4_niche import process as node4

state = node4(state)
print(f'\n--- Node 4 Output ---')
print(f'Niches identified : {state["n_niches"]}')
print()
for n in state['niches']:
    print(f'  Niche {n["cluster_id"]}  [{n["bio_label"]:25s}]  '
          f'tiles={n["n_tiles"]:3d}  Z={n["dom_z_score"]:+.3f}')
print()
print(state['niche_summary'])

In [ ]:
import matplotlib.patches as mpatches

niches     = state['niches']
km_labels  = np.array(state['kmeans_labels'])
tiles      = state['tiles']
wsi_dims   = state['wsi_dims']
pw_hm      = state['pathway_heatmap']

# Colour per niche
palette = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
niche_colors = {n['cluster_id']: palette[i % len(palette)] for i, n in enumerate(niches)}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: spatial niche map
for i, tile in enumerate(tiles):
    c = niche_colors.get(km_labels[i], '#cccccc')
    niche_obj = next((n for n in niches if n['cluster_id'] == km_labels[i]), None)
    axes[0].scatter(tile['x'] / wsi_dims[0], 1 - tile['y'] / wsi_dims[1],
                    c=c, s=25, alpha=0.7, edgecolors='none')

# Centroids
for n in niches:
    axes[0].scatter(n['centroid_x'] / wsi_dims[0], 1 - n['centroid_y'] / wsi_dims[1],
                    c=niche_colors[n['cluster_id']], s=200, marker='*',
                    edgecolors='black', linewidths=0.8, zorder=5)
    axes[0].text(n['centroid_x'] / wsi_dims[0] + 0.01,
                 1 - n['centroid_y'] / wsi_dims[1],
                 n['bio_label'], fontsize=7)

patches = [mpatches.Patch(color=niche_colors[n['cluster_id']], label=f"N{n['cluster_id']}: {n['bio_label']}")
           for n in niches]
axes[0].legend(handles=patches, fontsize=7, loc='upper right')
axes[0].set_title('Spatial Niche Map (normalised coordinates)')
axes[0].set_xlabel('X (fraction of WSI width)')
axes[0].set_ylabel('Y (fraction of WSI height)')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

# Right: pathway Z-score heatmap
pathways  = list(pw_hm.keys())
n_ids     = [str(n['cluster_id']) for n in niches]
hm_mat    = np.array([[pw_hm[pw].get(nid, 0.0) for nid in n_ids] for pw in pathways])

im = axes[1].imshow(hm_mat, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(n_ids)))
axes[1].set_xticklabels([f"N{n['cluster_id']}\n{n['bio_label'][:12]}" for n in niches], fontsize=7)
axes[1].set_yticks(range(len(pathways)))
axes[1].set_yticklabels([p.replace('_',' ') for p in pathways], fontsize=8)
axes[1].set_title('Pathway Activity Z-scores per Niche')
plt.colorbar(im, ax=axes[1], label='Z-score')

plt.suptitle('Node 4 — Spatial Niche Construction (DBSCAN + KMeans)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8 — Summary Dashboard

In [ ]:
from IPython.display import display, HTML

dominant_niche = state['niches'][0]
dominant_tp    = state['dominant_program_per_tp'][-1]

html = f"""
<div style='font-family:sans-serif; border:2px solid #667eea; border-radius:10px; padding:24px; max-width:700px'>
  <h2 style='color:#667eea; margin-top:0'>Pipeline Complete ✅</h2>
  <p style='color:#555'>TCGA-HZ-7926 &middot; Pancreatic Adenocarcinoma &middot; H&amp;E WSI</p>
  <table style='width:100%; border-collapse:collapse'>
    <tr style='background:#f8f9fa'>
      <th style='padding:8px; text-align:left; border-bottom:1px solid #ddd'>Metric</th>
      <th style='padding:8px; text-align:right; border-bottom:1px solid #ddd'>Value</th>
    </tr>
    <tr><td style='padding:8px'>Tissue tiles extracted</td>
        <td style='padding:8px; text-align:right'><b>{state['n_tiles']}</b></td></tr>
    <tr style='background:#f8f9fa'>
        <td style='padding:8px'>Nuclei detected (30 patches)</td>
        <td style='padding:8px; text-align:right'><b>{state['n_cells']:,}</b></td></tr>
    <tr><td style='padding:8px'>Protein channels inferred</td>
        <td style='padding:8px; text-align:right'><b>{state['n_channels']}</b></td></tr>
    <tr style='background:#f8f9fa'>
        <td style='padding:8px'>Immune score (T-cell panel)</td>
        <td style='padding:8px; text-align:right'><b>{state['immune_score']:.3f}</b></td></tr>
    <tr><td style='padding:8px'>Stromal score (fibrosis panel)</td>
        <td style='padding:8px; text-align:right'><b>{state['stromal_score']:.3f}</b></td></tr>
    <tr style='background:#f8f9fa'>
        <td style='padding:8px'>Spatiotemporal niches</td>
        <td style='padding:8px; text-align:right'><b>{state['n_niches']}</b></td></tr>
    <tr><td style='padding:8px'>Dominant niche</td>
        <td style='padding:8px; text-align:right'><b>{dominant_niche['bio_label']}</b> (Z={dominant_niche['dom_z_score']:+.3f})</td></tr>
    <tr style='background:#f8f9fa'>
        <td style='padding:8px'>Late-stage dominant immune state</td>
        <td style='padding:8px; text-align:right'><b>{dominant_tp['dominant']}</b> ({dominant_tp['fraction']:.3f})</td></tr>
  </table>
  <p style='margin-top:16px; color:#333'>{state['niche_summary']}</p>
  <p style='color:#888; font-size:12px'>Manuscript under preparation &middot; Contact: bbh@imdlab.org</p>
</div>
"""
display(HTML(html))